# GraphRAG Retrieval for Swiss Legal Citations

This notebook implements a **GraphRAG-enhanced retrieval pipeline** for the Omnilex legal citation competition.

## How GraphRAG works
1. **Indexing**: An LLM extracts entities (laws, courts, legal concepts) and relationships from documents, builds a knowledge graph, runs community detection, and writes community summaries.
2. **Local search**: Finds specific entities via vector similarity then expands through graph neighborhoods.
3. **Global search**: Answers broad questions by aggregating community summaries.

## Why it fits Swiss legal retrieval
- Law articles explicitly reference other articles — this becomes graph edges.
- Court decisions cite statutes and prior decisions — captured as relationships.
- Community detection groups related legal provisions (e.g. contract law cluster).
- Multi-hop traversal surfaces indirectly-cited provisions BM25 would miss.

## Setup required before running
1. Edit `graphrag/.env` with your API credentials.
2. Run data preparation: `python scripts/prepare_graphrag_data.py`
3. Run indexing: `graphrag index --root ./graphrag`
4. Then execute this notebook.

## 1. Configuration

In [ ]:
import os
import sys
from pathlib import Path

DATASET_MODE = "val"          # "val" for evaluation, "test" for submission
FORCE_REBUILD_INDICES = False  # Set True to rebuild BM25 indices
USE_HYBRID = True              # Combine GraphRAG + BM25 results

KAGGLE_ENV = "KAGGLE_KERNEL_RUN_TYPE" in os.environ

if KAGGLE_ENV:
    DATA_PATH = Path("/kaggle/input/omnilex-data")
    MODEL_PATH = Path("/kaggle/input/llama-model")
    OUTPUT_PATH = Path("/kaggle/working")
    INDEX_PATH = Path("/kaggle/input/omnilex-indices")
    GRAPHRAG_ROOT = Path("/kaggle/input/omnilex-graphrag")
    sys.path.insert(0, "/kaggle/input/omnilex-utils")
else:
    REPO_ROOT = Path(".").resolve().parent
    DATA_PATH = REPO_ROOT / "data" / "llm-agentic-legal-information-retrieval"
    MODEL_PATH = REPO_ROOT / "models"
    OUTPUT_PATH = REPO_ROOT / "output"
    INDEX_PATH = REPO_ROOT / "data" / "processed"
    GRAPHRAG_ROOT = REPO_ROOT / "graphrag"

LAWS_CSV = DATA_PATH / "laws_de.csv"
COURTS_CSV = DATA_PATH / "court_considerations.csv"
LAWS_INDEX_PATH = INDEX_PATH / "laws_index.pkl"
COURTS_INDEX_PATH = INDEX_PATH / "courts_index.pkl"
QUERY_FILE = DATA_PATH / f"{DATASET_MODE}.csv"

OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
INDEX_PATH.mkdir(parents=True, exist_ok=True)

print(f"Environment: {'Kaggle' if KAGGLE_ENV else 'Local'}")
print(f"Dataset mode: {DATASET_MODE}")
print(f"GraphRAG root: {GRAPHRAG_ROOT}")
graphrag_output = GRAPHRAG_ROOT / "output"
print(f"GraphRAG index ready: {(graphrag_output / 'entities.parquet').exists()}")

## 2. Verify GraphRAG Index

If the index is not ready, follow the instructions below to build it.

In [ ]:
GRAPHRAG_OUTPUT = GRAPHRAG_ROOT / "output"

expected_files = [
    "entities.parquet",
    "communities.parquet",
    "community_reports.parquet",
    "text_units.parquet",
    "relationships.parquet",
]

missing = [f for f in expected_files if not (GRAPHRAG_OUTPUT / f).exists()]

if missing:
    print("GraphRAG index NOT ready. Missing files:")
    for f in missing:
        print(f"  - {f}")
    print()
    print("To build the index, run these commands from the repo root:")
    print("  1. Edit graphrag/.env with your API credentials")
    print("  2. python scripts/prepare_graphrag_data.py --laws-only  # Start with just laws")
    print("  3. graphrag index --root ./graphrag")
    print()
    print("For the full corpus (laws + courts, may take hours):")
    print("  2. python scripts/prepare_graphrag_data.py")
    print("  3. graphrag index --root ./graphrag")
    GRAPHRAG_READY = False
else:
    import pandas as pd
    entities_df = pd.read_parquet(GRAPHRAG_OUTPUT / "entities.parquet")
    communities_df = pd.read_parquet(GRAPHRAG_OUTPUT / "communities.parquet")
    reports_df = pd.read_parquet(GRAPHRAG_OUTPUT / "community_reports.parquet")
    print("GraphRAG index is ready!")
    print(f"  Entities:            {len(entities_df):,}")
    print(f"  Communities:         {len(communities_df):,}")
    print(f"  Community reports:   {len(reports_df):,}")
    print()
    print("Top entity types:")
    if "type" in entities_df.columns:
        print(entities_df["type"].value_counts().head(10).to_string())
    GRAPHRAG_READY = True

## 3. Initialize GraphRAG Search Tools

In [ ]:
if not GRAPHRAG_READY:
    print("Skipping — GraphRAG index not built yet.")
else:
    # Add repo src to path if running locally
    if not KAGGLE_ENV:
        sys.path.insert(0, str(REPO_ROOT / "src"))

    from omnilex.retrieval.graphrag_tools import (
        GraphRAGGlobalSearchTool,
        GraphRAGLocalSearchTool,
        GraphRAGSearchEngine,
    )

    local_tool = GraphRAGLocalSearchTool(
        graphrag_root=GRAPHRAG_ROOT,
        community_level=2,
        response_type="Single Paragraph",
    )
    global_tool = GraphRAGGlobalSearchTool(
        graphrag_root=GRAPHRAG_ROOT,
        dynamic_community_selection=True,
        response_type="Multiple Paragraphs",
    )

    print("GraphRAG tools initialised (lazy-loaded — first query will load parquets).")

## 4. Build / Load BM25 Indices

In [ ]:
import pickle
import re

import pandas as pd
from rank_bm25 import BM25Okapi
from tqdm.notebook import tqdm


class BM25Index:
    def __init__(self, documents=None, text_field="text", citation_field="citation"):
        self.text_field = text_field
        self.citation_field = citation_field
        self.documents = []
        self.index = None
        self._tokenized_corpus = []
        if documents:
            self.build(documents)

    def tokenize(self, text):
        return [t for t in re.split(r"\W+", text.lower()) if t]

    def build(self, documents):
        self.documents = documents
        self._tokenized_corpus = [self.tokenize(d.get(self.text_field, "")) for d in documents]
        self.index = BM25Okapi(self._tokenized_corpus)

    def search(self, query, top_k=10, return_scores=False):
        if self.index is None:
            return []
        tokens = self.tokenize(query)
        if not tokens:
            return []
        scores = self.index.get_scores(tokens)
        top_idx = scores.argsort()[-top_k:][::-1]
        results = []
        for i in top_idx:
            if scores[i] <= 0:
                continue
            doc = self.documents[i].copy()
            if return_scores:
                doc["_score"] = float(scores[i])
            results.append(doc)
        return results

    def save(self, path):
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        with open(path, "wb") as f:
            pickle.dump(
                {"documents": self.documents, "tokenized_corpus": self._tokenized_corpus,
                 "text_field": self.text_field, "citation_field": self.citation_field}, f
            )

    @classmethod
    def load(cls, path):
        with open(path, "rb") as f:
            data = pickle.load(f)
        inst = cls(text_field=data["text_field"], citation_field=data.get("citation_field", "citation"))
        inst.documents = data["documents"]
        inst._tokenized_corpus = data["tokenized_corpus"]
        inst.index = BM25Okapi(inst._tokenized_corpus)
        return inst


def load_csv_corpus(csv_path, chunk_size=100_000, max_rows=None):
    documents = []
    rows_loaded = 0
    for chunk in pd.read_csv(csv_path, chunksize=chunk_size):
        for _, row in chunk.iterrows():
            if max_rows and rows_loaded >= max_rows:
                return documents
            documents.append({"citation": str(row["citation"]), "text": str(row["text"]) if pd.notna(row["text"]) else ""})
            rows_loaded += 1
    return documents


def get_or_build_index(name, csv_path, index_path, force_rebuild=False, max_rows=None):
    if index_path.exists() and not force_rebuild:
        print(f"Loading cached {name} index...")
        idx = BM25Index.load(index_path)
        print(f"  {len(idx.documents):,} documents")
        return idx
    if not csv_path.exists():
        print(f"Warning: {csv_path} not found.")
        return BM25Index(documents=[])
    print(f"Building {name} index from CSV...")
    docs = load_csv_corpus(csv_path, max_rows=max_rows)
    idx = BM25Index(documents=docs)
    if not KAGGLE_ENV:
        idx.save(index_path)
    return idx


laws_index = get_or_build_index("laws", LAWS_CSV, LAWS_INDEX_PATH, FORCE_REBUILD_INDICES)
courts_index = get_or_build_index("courts", COURTS_CSV, COURTS_INDEX_PATH, FORCE_REBUILD_INDICES, max_rows=100_000)

print(f"Laws index:  {len(laws_index.documents):,} documents")
print(f"Courts index: {len(courts_index.documents):,} documents")

## 5. BM25 Search Tools

In [ ]:
class LawSearchTool:
    name = "search_laws"
    description = """Search Swiss federal laws by keywords."""

    def __init__(self, index, top_k=40, max_excerpt=300):
        self.index = index
        self.top_k = top_k
        self.max_excerpt = max_excerpt
        self._last_results = []

    def __call__(self, query):
        return self.run(query)

    def run(self, query):
        if not query.strip():
            return "Error: empty query"
        self._last_results = self.index.search(query, top_k=self.top_k)
        if not self._last_results:
            return f"No laws found for: '{query}'"
        lines = []
        for doc in self._last_results:
            text = doc.get("text", "")[:self.max_excerpt]
            lines.append(f"- {doc.get('citation', '')}: {text}")
        return "\n".join(lines)

    def get_last_citations(self):
        return [d.get("citation", "") for d in self._last_results if d.get("citation")]


class CourtSearchTool:
    name = "search_courts"
    description = """Search Swiss Federal Court decisions by keywords."""

    def __init__(self, index, top_k=40, max_excerpt=300):
        self.index = index
        self.top_k = top_k
        self.max_excerpt = max_excerpt
        self._last_results = []

    def __call__(self, query):
        return self.run(query)

    def run(self, query):
        if not query.strip():
            return "Error: empty query"
        self._last_results = self.index.search(query, top_k=self.top_k)
        if not self._last_results:
            return f"No court decisions found for: '{query}'"
        lines = []
        for doc in self._last_results:
            text = doc.get("text", "")[:self.max_excerpt]
            lines.append(f"- {doc.get('citation', '')}: {text}")
        return "\n".join(lines)

    def get_last_citations(self):
        return [d.get("citation", "") for d in self._last_results if d.get("citation")]


law_tool = LawSearchTool(laws_index)
court_tool = CourtSearchTool(courts_index)
print("BM25 search tools ready.")

## 6. Test GraphRAG Search

Compare GraphRAG vs BM25 on a sample legal question.

In [ ]:
sample_query = "Welche Voraussetzungen gelten für den Abschluss eines gültigen Vertrags nach Schweizer Recht?"

print("=" * 60)
print(f"Query: {sample_query[:100]}")

print("\n--- BM25 Law Results ---")
print(law_tool.run("Vertrag Abschluss Voraussetzungen")[:800])

if GRAPHRAG_READY:
    print("\n--- GraphRAG Local Search ---")
    print(local_tool.run(sample_query)[:800])

    print("\n--- GraphRAG Global Search ---")
    print(global_tool.run("Voraussetzungen gültiger Vertrag Schweizer Recht")[:800])

## 7. Combined Retrieval Function

Uses GraphRAG local search + BM25 to collect candidate citations for each query.

In [ ]:
import re


def extract_citations_from_text(text: str) -> list[str]:
    """Extract Swiss legal citation patterns from text."""
    citations = []

    # BGE citations: BGE 139 I 2 E. 1
    citations += re.findall(
        r"BGE\s+\d{1,3}\s+[IVX]+[a-z]?\s+\d+(?:\s+E\.\s*\d+[a-z]?(?:\.\d+)?)?",
        text,
    )

    # Art. citations: Art. 1 OR, Art. 104 Abs. 2 ZGB
    citations += re.findall(
        r"Art\.\s+\d+[a-z]?(?:\s+Abs\.?\s*\d+)?\s+[A-Z]{2,6}",
        text,
    )

    # SR citations: SR 220, SR 210
    citations += re.findall(r"SR\s+\d{3}(?:\.\d+)?", text)

    return list(set(citations))


def hybrid_retrieve(query: str, use_graphrag: bool = True) -> list[str]:
    """Retrieve citations using BM25 and optionally GraphRAG.

    Returns a deduplicated list of citation strings.
    """
    citations = []

    # BM25 retrieval — always available
    law_tool.run(query)
    citations += law_tool.get_last_citations()

    court_tool.run(query)
    citations += court_tool.get_last_citations()

    # GraphRAG local search — if index is available
    if use_graphrag and GRAPHRAG_READY:
        graph_answer = local_tool.run(query)
        # Extract any explicit citations embedded in the answer text
        citations += extract_citations_from_text(graph_answer)

    return list(set(c for c in citations if c.strip()))


# Quick test
test_citations = hybrid_retrieve(
    "Vertrag Abschluss Voraussetzungen",
    use_graphrag=GRAPHRAG_READY and USE_HYBRID,
)
print(f"Found {len(test_citations)} unique citations:")
for c in test_citations[:10]:
    print(f"  - {c}")

## 8. Load Queries and Generate Predictions

In [ ]:
import pandas as pd
from tqdm.notebook import tqdm

if not QUERY_FILE.exists():
    raise FileNotFoundError(f"Query file not found: {QUERY_FILE}")

query_df = pd.read_csv(QUERY_FILE)
print(f"Loaded {len(query_df)} queries")
print(f"Columns: {list(query_df.columns)}")
query_df.head(2)

In [ ]:
predictions = []

for _, row in tqdm(query_df.iterrows(), total=len(query_df), desc="Retrieving citations"):
    query_id = row["query_id"]
    query_text = row["query"]

    found_citations = hybrid_retrieve(
        query_text,
        use_graphrag=GRAPHRAG_READY and USE_HYBRID,
    )

    predictions.append({
        "query_id": query_id,
        "predicted_citations": ";".join(found_citations),
    })

predictions_df = pd.DataFrame(predictions)
print(f"Generated {len(predictions_df)} predictions")
predictions_df.head()

## 9. Save Submission

In [ ]:
mode_tag = "graphrag" if GRAPHRAG_READY and USE_HYBRID else "bm25"
submission_path = OUTPUT_PATH / f"submission_{mode_tag}_{DATASET_MODE}.csv"
predictions_df.to_csv(submission_path, index=False)
print(f"Submission saved: {submission_path}")

## 10. Local Evaluation (validation mode only)

In [ ]:
if DATASET_MODE == "val" and "gold_citations" in query_df.columns:
    from collections.abc import Sequence

    def citation_f1(predicted, gold):
        pred_set, gold_set = set(predicted), set(gold)
        if not pred_set and not gold_set:
            return {"precision": 1.0, "recall": 1.0, "f1": 1.0}
        if not pred_set:
            return {"precision": 0.0, "recall": 0.0, "f1": 0.0}
        if not gold_set:
            return {"precision": 0.0, "recall": 1.0, "f1": 0.0}
        tp = len(pred_set & gold_set)
        p = tp / len(pred_set)
        r = tp / len(gold_set)
        f1 = 2 * p * r / (p + r) if p + r else 0.0
        return {"precision": p, "recall": r, "f1": f1}

    merged = predictions_df.merge(query_df[["query_id", "gold_citations"]], on="query_id")

    def parse(s):
        return [c.strip() for c in str(s).split(";") if c.strip()] if s else []

    preds = [parse(r["predicted_citations"]) for _, r in merged.iterrows()]
    golds = [parse(r["gold_citations"]) for _, r in merged.iterrows()]

    scores = [citation_f1(p, g) for p, g in zip(preds, golds)]
    n = len(scores)

    macro_f1 = sum(s["f1"] for s in scores) / n
    macro_p = sum(s["precision"] for s in scores) / n
    macro_r = sum(s["recall"] for s in scores) / n

    print("=" * 50)
    print("EVALUATION RESULTS")
    print("=" * 50)
    print(f"Queries evaluated: {n}")
    print(f"Mode: {'GraphRAG + BM25' if GRAPHRAG_READY and USE_HYBRID else 'BM25 only'}")
    print()
    print(f"Macro F1 (PRIMARY): {macro_f1:.4f}")
    print(f"Macro Precision:    {macro_p:.4f}")
    print(f"Macro Recall:       {macro_r:.4f}")
else:
    print("Evaluation skipped (test mode or no gold labels).")

## GraphRAG Indexing Reference

```bash
# Step 1 — edit credentials
nano graphrag/.env

# Step 2 — prepare input files (laws only for a quick first run)
python scripts/prepare_graphrag_data.py --laws-only

# Step 2b — full corpus (takes hours; needs 50GB+ disk)
python scripts/prepare_graphrag_data.py

# Step 3 — run indexing
graphrag index --root ./graphrag

# Step 4 — optional: tune prompts for Swiss law domain
graphrag prompt-tune --root ./graphrag
```

## Expected indexing costs (rough estimates with gpt-4o-mini)

| Corpus | Documents | Est. time | Est. cost |
|--------|-----------|-----------|----------|
| Laws only | ~1,000 statute files | 30–60 min | ~$5–10 |
| Full corpus | ~2,000 files | 4–8 hours | ~$30–60 |

Costs vary by provider and model. Using a cheaper model (e.g. `gpt-4o-mini`) reduces cost significantly.